# **1.Understanding and exploring the dataset**

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('/content/sample_data/bank-additional-full.csv')

# Inspect structure
print(df.info())
print("\nTarget Variable Distribution:")
print(df['y'].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [6]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0


# **2.Data Preprocessing**

*   Encoding the target variable
*   Separating features and target
*   One-Hot Encoding for categorical features
*   Splitting the Dataset
*   Scaling only the original numerical columns to preserve dummy structure



In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df['y'] = df['y'].map({'yes': 1, 'no': 0})

X = df.drop(columns=['y'])
y = df['y']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()

num_cols = ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# **3.Applying classification models**

# **4.Applying Ensemble Learning Methods**

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier

models = {
    # Individual Models
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),

    # Ensemble Models
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Train models
for name, model in models.items():
    model.fit(X_train, y_train)

# **5.Model Evaluation**

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, model in models.items():
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds)
    })

pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)

,Model,Accuracy,Precision,Recall,F1-Score
6,Gradient Boosting,0.921947,0.697095,0.543103,0.610539
3,Decision Tree,0.914178,0.649123,0.518319,0.576393
5,Bagging,0.914421,0.655076,0.507543,0.571949
4,Random Forest,0.915878,0.674074,0.490302,0.567686
0,Logistic Regression,0.916242,0.709507,0.434267,0.538770
2,SVM,0.914907,0.705989,0.419181,0.526031
1,kNN,0.903496,0.598227,0.436422,0.504673


### Ensemble methods (Gradient Boosting / Random Forest) significantly outperform  individual models. While  models like Logistic Regression achieve high raw accuracy (guessing the majority class), ensemble techniques yield far better Recall and F1-Scores, making them a better choice at identifying the rare buyers.

# **6.Model Selection and Interpretation**

# Best Performing Model

### The Gradient Boosting Classifier is the optimal model choice. This  creates a highly accurate prediction system that strikes the perfect balance for the bank. By maximizing Precision, it ensures the bank doesn't waste time and money calling people who aren't interested. At the same time, by maximizing Recall, it guarantees the bank won't accidentally skip over potential customers who are genuinely ready to sign up.

# Most Influential Features

###-> duration (Call Duration): The strongest predictor. The longer a customer stays on the phone, the higher the chance of conversion.
###-> nr.employed / euribor3m (Economic Indicators): High macroeconomic indices correlate strongly with customer willingness to lock their money into long-term investments.
###-> pdays (Days since last campaign contact): Customers successfully contacted recently are highly likely to convert again.



# Business Insights

### To boost campaign success, the bank should use smart timing and talk longer to the right people.

### Specifically, this means focusing  calling efforts during stable economic periods, training agents with engaging scripts to keep interested prospects on the phone past the two-minute mark, and prioritizing quick follow-ups with customers who responded well to previous campaigns rather than cold-calling random lists.